# Node · 30-minute Colab conversation experiment

Continue the existing Node checkpoint on an assigned Colab GPU. **No pretrained model is substituted.** GPU hardware and availability depend on Colab; no purchase or upgrade is required by this notebook.

1. Select **Runtime → Change runtime type → GPU** (use a free option if available).
2. Run the cells in order. The training stage lasts **1,800 seconds**; before/after evaluation adds time.
3. Inspect both practiced and unseen prompts. The script retains a **best-validation** checkpoint as well as final weights; longer training can overfit.
4. Download the results ZIP, then disconnect/delete the runtime to release resources.

The model still has only a **256-byte context**. This notebook does not promise general fluency or reliable coding.

In [ ]:
from pathlib import Path
import subprocess, sys, json
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU assigned. Select Runtime > Change runtime type > GPU and reconnect. Training has not started.")
print("Assigned GPU:", torch.cuda.get_device_name(0))
# Keep Colab's existing CUDA-enabled PyTorch. Do NOT install the CPU-only requirements.txt.


In [ ]:
ROOT = Path("/content/node-ai-colab")
REPO = "https://github.com/ItzJoker2YT-city/node-ai.git"
if ROOT.exists():
    raise RuntimeError("This project folder already exists. Reuse your existing run or choose a new empty folder; nothing was overwritten.")
subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
assert (ROOT / "colab/train_colab.py").is_file()
print("Source commit:", subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip())


## Run the 30-minute experiment
The cell below performs real CUDA training. Logs show elapsed seconds, steps, assigned hardware, and held-out validation loss. A baseline is saved before any update. Checkpoints are stored in the Colab runtime, **not Google Drive**.

Do not rerun this cell while it is still executing. If the free GPU is unavailable, do not buy compute just to run this notebook.

In [ ]:
command = [sys.executable, str(ROOT / "colab/train_colab.py"),
           "--seconds", "1800", "--device", "cuda", "--batch-size", "16"]
subprocess.run(command, cwd=ROOT, check=True)


In [ ]:
OUTPUT = ROOT / "colab_outputs"
report = json.loads((OUTPUT / "evaluation.json").read_text())
print(json.dumps(report["run"], indent=2))
for before, after in zip(report["before"], report["after_best_validation"]):
    print("\nCASE:", before["case"])
    print("PROMPT:", before["prompt"])
    print("BEFORE:", before["output"])
    print("BEST-VALIDATION CHECKPOINT:", after["output"])
print("\nCAUTION:", report["note"])


## Download the results
`best.pt` is the best checkpoint by the fixed held-out validation windows. `final.pt` is the end-of-run checkpoint. Copy the chosen checkpoint to your local project's `data/node.pt` **after backing up the old file and stopping the local server**. These weights are CPU-portable.

Practiced greetings can look good even when unseen questions fail. Read `evaluation.json` before deciding which weights to keep. The best checkpoint can be the unchanged original baseline.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/node-colab-results", "zip", OUTPUT)
print("Created:", archive)
files.download(archive)


## Release the runtime
After the ZIP has finished downloading, use **Runtime → Disconnect and delete runtime**. The repository and notebook do not create a public chat server, mount Drive, request secrets, or upload your Google account details to GitHub.